In [ ]:
# import pandas as pd
# import numpy as np
# import os
# from sklearn.preprocessing import MinMaxScaler
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, Dense
# from tensorflow.keras.optimizers import Adam

# # ۱. تنظیمات مسیرها
# file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
# output_path = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output4.xlsx'

# all_sensors = ['AssetID_9357', 'AssetID_9343', 'AssetID_9358', 'AssetID_9359', 
#                'AssetID_9360', 'AssetID_9361', 'AssetID_9368', 'AssetID_9369', 'AssetID_9370']

# # ۲. بارگذاری داده‌ها
# df = pd.read_excel(file_path)
# df['date'] = pd.to_datetime(df['date'])
# df = df.sort_values(by='date')

# df_clean = df.dropna(subset=all_sensors).copy()
# raw_data = df_clean[all_sensors].values

# # ۳. نرمال‌سازی
# scaler = MinMaxScaler()
# scaled_data = scaler.fit_transform(raw_data)

# # ۴. ساخت معماری اتوانکودر (همان ساختار قبلی)
# input_dim = len(all_sensors)
# input_layer = Input(shape=(input_dim,))
# encoded = Dense(16, activation='relu')(input_layer)
# latent_space = Dense(8, activation='relu')(encoded)
# decoded = Dense(16, activation='relu')(latent_space)
# output_layer = Dense(input_dim, activation='sigmoid')(decoded)

# autoencoder = Model(inputs=input_layer, outputs=output_layer)
# autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')

# # ۵. آموزش مدل (روی کل داده‌ها برای یادگیری دقیق رفتار سیستم)
# print("🚀 در حال یادگیری رفتار سیستم از کل داده‌ها...")
# autoencoder.fit(scaled_data, scaled_data, epochs=150, batch_size=32, shuffle=True, verbose=0)

# # ۶. محاسبه بازسازی و خطای آن
# reconstructed_data = autoencoder.predict(scaled_data)
# mse_errors = np.mean(np.power(scaled_data - reconstructed_data, 2), axis=1)

# # ۷. محاسبه آستانه ناهنجاری
# threshold = np.mean(mse_errors) + (3*np.std(mse_errors))

# # ۸. اضافه کردن نتایج به دیتافریم
# df_clean['Systematic_Deviation_Index'] = mse_errors
# df_clean['Anomaly_Threshold'] = threshold
# df_clean['Is_Anomaly'] = df_clean['Systematic_Deviation_Index'] > threshold

# # ================= بخش جدید: فیلتر کردن برای یک ماه آخر =================
# last_date = df_clean['date'].max()
# start_of_last_month = last_date - pd.Timedelta(days=30)
# df_last_month = df_clean[df_clean['date'] >= start_of_last_month].copy()
# # ======================================================================

# # ۹. ذخیره خروجی (فقط دیتای ماه آخر)
# os.makedirs(os.path.dirname(output_path), exist_ok=True)
# df_last_month.to_excel(output_path, index=False)

# print("-" * 40)
# print(f"✅ تحلیل با موفقیت انجام شد.")
# print(f"📅 بازه خروجی: از {df_last_month['date'].min()} تا {df_last_month['date'].max()}")
# print(f"🚨 تعداد ناهنجاری در ماه اخیر: {df_last_month['Is_Anomaly'].sum()}")
# print(f"📂 فایل (فقط ماه آخر) در مسیر زیر ذخیره شد:\n{output_path}")

🚀 در حال یادگیری رفتار سیستم از کل داده‌ها...
368/368 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
----------------------------------------
✅ تحلیل با موفقیت انجام شد.
📅 بازه خروجی: از 2026-04-04 08:31:20 تا 2026-05-04 05:16:35
🚨 تعداد ناهنجاری در ماه اخیر: 3
📂 فایل (فقط ماه آخر) در مسیر زیر ذخیره شد:
outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output4.xlsx


In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# غیرفعال کردن هشدارهای TensorFlow
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

def run_autoencoder_anomaly_detection():
    """اجرای تحلیل ناهنجاری با Autoencoder و ذخیره خروجی"""
    
    print("="*60)
    print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    # ۱. تنظیمات مسیرها
    file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
    output_path = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output4.xlsx'
    
    # ایجاد پوشه خروجی
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    all_sensors = ['AssetID_9357', 'AssetID_9343', 'AssetID_9358', 'AssetID_9359', 
                   'AssetID_9360', 'AssetID_9361', 'AssetID_9368', 'AssetID_9369', 'AssetID_9370']

    # ۲. بارگذاری داده‌ها
    try:
        df = pd.read_excel(file_path)
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values(by='date')
        print(f"✅ فایل اصلی با موفقیت خوانده شد. تعداد رکوردها: {len(df):,}")
        print(f"📅 بازه زمانی: {df['date'].min()} تا {df['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return None

    df_clean = df.dropna(subset=all_sensors).copy()
    raw_data = df_clean[all_sensors].values
    print(f"📊 تعداد رکوردهای بدون داده خالی: {len(df_clean):,}")

    # ۳. نرمال‌سازی
    print("🔄 مرحله 1: نرمال‌سازی داده‌ها...")
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(raw_data)
    print(f"   ✅ نرمال‌سازی با {len(all_sensors)} سنسور انجام شد")

    # ۴. ساخت معماری اتوانکودر
    print("🔄 مرحله 2: ساخت و آموزش مدل Autoencoder...")
    
    input_dim = len(all_sensors)
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(16, activation='relu')(input_layer)
    latent_space = Dense(8, activation='relu')(encoded)
    decoded = Dense(16, activation='relu')(latent_space)
    output_layer = Dense(input_dim, activation='sigmoid')(decoded)

    autoencoder = Model(inputs=input_layer, outputs=output_layer)
    autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')

    # ۵. آموزش مدل (روی کل داده‌ها برای یادگیری دقیق رفتار سیستم)
    print("   🚀 در حال یادگیری رفتار سیستم از کل داده‌ها...")
    autoencoder.fit(scaled_data, scaled_data, epochs=150, batch_size=32, shuffle=True, verbose=0)
    print("   ✅ آموزش مدل با موفقیت انجام شد")

    # ۶. محاسبه بازسازی و خطای آن
    print("🔄 مرحله 3: محاسبه خطای بازسازی...")
    reconstructed_data = autoencoder.predict(scaled_data, verbose=0)
    mse_errors = np.mean(np.power(scaled_data - reconstructed_data, 2), axis=1)
    print(f"   ✅ خطای بازسازی برای {len(mse_errors):,} رکورد محاسبه شد")

    # ۷. محاسبه آستانه ناهنجاری
    print("🔄 مرحله 4: محاسبه آستانه ناهنجاری...")
    threshold = np.mean(mse_errors) + (3*np.std(mse_errors))
    print(f"   ✅ آستانه ناهنجاری: {threshold:.6f}")
    print(f"   میانگین خطا: {np.mean(mse_errors):.6f}")
    print(f"   انحراف معیار خطا: {np.std(mse_errors):.6f}")

    # ۸. اضافه کردن نتایج به دیتافریم
    df_clean['Systematic_Deviation_Index'] = mse_errors
    df_clean['Anomaly_Threshold'] = threshold
    df_clean['Is_Anomaly'] = df_clean['Systematic_Deviation_Index'] > threshold
    df_clean['Anomaly_Label'] = df_clean['Is_Anomaly'].map({True: '⚠️ Anomaly', False: '✅ Normal'})

    # ================= بخش جدید: فیلتر کردن برای یک ماه آخر =================
    print("🔄 مرحله 5: فیلتر کردن داده‌های یک ماه اخیر...")
    
    last_date = df_clean['date'].max()
    start_of_last_month = last_date - pd.Timedelta(days=30)
    df_last_month = df_clean[df_clean['date'] >= start_of_last_month].copy()
    
    print(f"   📅 بازه خروجی: از {df_last_month['date'].min()} تا {df_last_month['date'].max()}")
    print(f"   تعداد رکوردهای ماه اخیر: {len(df_last_month):,}")
    print(f"   تعداد ناهنجاری در ماه اخیر: {df_last_month['Is_Anomaly'].sum():,}")
    print(f"   درصد ناهنجاری: {(df_last_month['Is_Anomaly'].sum()/len(df_last_month)*100):.2f}%")

    # ۹. ذخیره خروجی (فقط دیتای ماه آخر)
    print("💾 مرحله 6: ذخیره خروجی...")
    
    try:
        df_last_month.to_excel(output_path, index=False)
        print(f"✅ تحلیل با موفقیت انجام شد.")
        print(f"📂 فایل (فقط ماه آخر) در مسیر زیر ذخیره شد:\n{output_path}")
        print(f"📊 تعداد رکوردهای نهایی: {len(df_last_month):,}")
        print(f"📋 تعداد ستون‌ها: {len(df_last_month.columns)}")
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return None
    
    print("="*60)
    print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return df_last_month

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - پایش ناهنجاری با Autoencoder")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["22:28", "22:15", "22:16"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result = run_autoencoder_anomaly_detection()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه پایش ناهنجاری با Autoencoder")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")


🚀 شروع برنامه پایش ناهنجاری با Autoencoder
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - پایش ناهنجاری با Autoencoder
⏰ زمان‌های اجرا (هر روز):
   - ساعت 10:00
   - ساعت 10:05
   - ساعت 10:10
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-01 22:28:06
🔄 شروع تحلیل در 2026-07-01 22:28:06
✅ فایل اصلی با موفقیت خوانده شد. تعداد رکوردها: 11,752
📅 بازه زمانی: 2021-03-16 05:33:48 تا 2026-05-04 05:16:35
📊 تعداد رکوردهای بدون داده خالی: 11,752
🔄 مرحله 1: نرمال‌سازی داده‌ها...
   ✅ نرمال‌سازی با 9 سنسور انجام شد
🔄 مرحله 2: ساخت و آموزش مدل Autoencoder...
   🚀 در حال یادگیری رفتار سیستم از کل داده‌ها...
   ✅ آموزش مدل با موفقیت انجام شد
🔄 مرحله 3: محاسبه خطای بازسازی...
   ✅ خطای بازسازی برای 11,752 رکورد محاسبه شد
🔄 مرحله 4: محاسبه آستانه ناهنجاری...
   ✅ آستانه ناهنجاری: 0.005356
   میانگین خطا: 0.000381
   انحراف معیار خطا: 0.001658
🔄 مرحله 5: فیلتر کردن داده‌های یک ماه اخیر...
   📅 بازه خروجی: از 2026-04-04 08:31:20 تا 2026-05-04 05:16:35
   تعداد رکوردهای ماه اخیر: 152
   